# Combined Pipeline — Baseline + DA-Fusion + CEMS + (partial) CHARMS

Single Kaggle-ready notebook that runs baseline + DA-Fusion + CEMS (+ partial
CHARMS) with flip4x augmentation, full GroupKFold CV, per-fold OOF predictions,
and a Kaggle submission CSV. All method additions are gated by config flags so
the notebook supports ablations (all flags False → baseline).

## CHARMS caveat (read before running with USE_CHARMS=True)

The standalone CHARMS notebook (`src/methods/charms/charms_kaggle.ipynb`) uses a
SmallCNN backbone and computes Optimal-Transport alignment between conv-channel
similarity matrices and tabular-attribute embeddings. That full stack is **not
directly transferable** to a frozen-DINOv2 + MLP pipeline — the OT step requires
spatial conv channels that don't exist in our 32-d latent.

What this notebook ports, when `USE_CHARMS=True`, is the **i2t auxiliary loss**
only (Option B in `integration_plan.md`): one linear head per latent dimension
(32 heads in total) each predicts all 5 targets from a single scalar latent
activation, and their weighted SmoothL1 losses are added to the main loss with
weight `cfg.charms_i2t_weight` (default 0.3). The OT Sinkhorn alignment and the
TabularBranch are **not** implemented here. For the full CHARMS, run its
standalone notebook and use `ensemble_charms.py` to blend submissions.

## Method toggles (ablation matrix)

| USE_DA_FUSION | USE_CEMS | USE_CHARMS | Configuration                                |
|---------------|----------|------------|----------------------------------------------|
| False         | False    | False      | Plain baseline                               |
| True          | False    | False      | DA-Fusion only (matches kaggle_da_fusion)    |
| False         | True     | False      | CEMS only   (matches kaggle_cems_pipeline)   |
| False         | False    | True       | Baseline + i2t auxiliary loss                |
| True          | True     | True       | Combined (full stack)                        |

The baseline split (`GroupKFold(n_splits=5)`, SEED=42), submission CSV format,
and weighted-SmoothL1 loss are preserved exactly from `kaggle_baseline_pipeline.ipynb`.


## 1. Setup & Imports

In [ ]:
import os, math, random, time, json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")


## 2. Configuration

All method toggles, smoke-test toggles, and hyperparameters live here. For an
ablation, flip the `USE_*` flags; for a Kaggle smoke test (1 fold, 1 epoch),
set `cfg.SMOKE_TEST=True`.

In [ ]:
cfg = SimpleNamespace(
    # --- Method toggles (all True = combined run; all False = baseline) ---
    USE_DA_FUSION=True,
    USE_CEMS=True,
    USE_CHARMS=True,       # partial port: i2t auxiliary loss on 32-d latent

    # --- Dry-run controls ---
    SMOKE_TEST=False,              # True overrides NUM_EPOCHS=1, NUM_FOLDS=1
    NUM_EPOCHS=80,
    NUM_FOLDS=5,
    MAX_ANCHORS_PER_EPOCH=None,    # CEMS: None = no cap (use all train anchors)

    # --- Paths (Kaggle layout) ---
    dataset_dir="/kaggle/input/competitions/csiro-biomass",
    dino_weights_dir="/kaggle/input/datasets/darealvictorslorer/dinov2-small-weights/dinov2-small",
    synthetic_dir="/kaggle/input/da-fusion-synthetic-biomass",
    synthetic_csv="/kaggle/input/da-fusion-synthetic-biomass/train_augmented.csv",
    output_dir="/kaggle/working",

    # --- Model ---
    input_dim=384, latent_dim=32, output_dim=5, dropout=0.1,

    # --- Training ---
    lr=3e-4, weight_decay=1e-3, batch_size=32, seed=42,

    # --- CEMS (ignored if USE_CEMS=False) ---
    sigma=1e-3, cems_method=1, initial_d=5, use_hessian=False,

    # --- CHARMS-i2t (ignored if USE_CHARMS=False) ---
    charms_i2t_weight=0.3,

    # --- Real-image flip4x augmentation ---
    use_flip4x=True,
)

if cfg.SMOKE_TEST:
    cfg.NUM_EPOCHS = 1
    cfg.NUM_FOLDS  = 1
    print("[SMOKE_TEST] NUM_EPOCHS=1, NUM_FOLDS=1")

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1,
           "GDM_g": 0.2, "Dry_Total_g": 0.5}

TRAIN_CSV     = os.path.join(cfg.dataset_dir, "train.csv")
TEST_CSV      = os.path.join(cfg.dataset_dir, "test.csv")
TRAIN_IMG_DIR = os.path.join(cfg.dataset_dir, "train")
TEST_IMG_DIR  = os.path.join(cfg.dataset_dir, "test")

print("Flags:  USE_DA_FUSION=", cfg.USE_DA_FUSION,
      " USE_CEMS=", cfg.USE_CEMS,
      " USE_CHARMS=", cfg.USE_CHARMS)
print("NUM_EPOCHS:", cfg.NUM_EPOCHS, " NUM_FOLDS:", cfg.NUM_FOLDS)
for p, label in [(TRAIN_CSV, "train.csv"), (TEST_CSV, "test.csv"),
                  (TRAIN_IMG_DIR, "train dir"), (TEST_IMG_DIR, "test dir"),
                  (cfg.dino_weights_dir, "dino dir"),
                  (cfg.synthetic_dir, "synth dir"),
                  (cfg.synthetic_csv, "synth csv")]:
    print(f"  {label:>12}: {os.path.exists(p)}  ({p})")


## 3. Load DINOv2 (frozen backbone)

In [ ]:
print(f"Loading DINOv2 ViT-S/14 from {cfg.dino_weights_dir} ...")
dino = AutoModel.from_pretrained(cfg.dino_weights_dir).eval().to(device)
for p in dino.parameters():
    p.requires_grad_(False)

_dummy = torch.zeros(1, 3, 252, 504, device=device)
with torch.no_grad():
    _out = dino(pixel_values=_dummy, interpolate_pos_encoding=True)
    _cls = _out.last_hidden_state[:, 0, :]
assert _cls.shape == (1, 384), f"Expected (1, 384), got {_cls.shape}"
print(f"DINOv2 smoke test passed — CLS shape: {_cls.shape}")
del _dummy, _out, _cls


## 4. Load Training CSV (real images)

Pivot long-format CSV to wide format (one row per image) and extract Y matrix.

In [ ]:
df_raw = pd.read_csv(TRAIN_CSV)
df_raw["image_id"] = df_raw["sample_id"].str.split("__").str[0]

df_wide = df_raw.pivot_table(
    index=["image_id", "image_path"],
    columns="target_name",
    values="target",
).reset_index()

Y_all_real          = df_wide[TARGETS].values.astype(np.float32)
train_image_ids_all = df_wide["image_id"].values

print(f"Training images: {len(df_wide)}")
print(f"Y_all_real shape: {Y_all_real.shape}")


## 5. Feature Extraction — DINOv2 CLS tokens

Preprocessing matches baseline exactly (504×252 resize, ImageNet norm). If `cfg.use_flip4x=True`, real images are extracted at 4 orientations so each image contributes 4 rows to `X_real`. Synthetic images (when `USE_DA_FUSION=True`) are extracted at a single orientation — they are already augmentations.

In [ ]:
_dino_transform = T.Compose([
    T.Resize((252, 504)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

_ORIENTATIONS = [
    lambda img: img,
    lambda img: TF.hflip(img),
    lambda img: TF.vflip(img),
    lambda img: TF.hflip(TF.vflip(img)),
]

def _extract_one(path, flip_fn=None):
    img = Image.open(path).convert("RGB")
    if flip_fn is not None:
        img = flip_fn(img)
    x = _dino_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out  = dino(pixel_values=x, interpolate_pos_encoding=True)
        feat = out.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()
    return feat

def extract_features(image_paths, augment=False, log_prefix=""):
    """Return (N, 384) if augment=False, (N*4, 384) if augment=True."""
    feats = []
    for i, p in enumerate(image_paths):
        if augment:
            for flip_fn in _ORIENTATIONS:
                feats.append(_extract_one(p, flip_fn))
        else:
            feats.append(_extract_one(p))
        if (i + 1) % 50 == 0:
            print(f"  {log_prefix}{i+1}/{len(image_paths)}")
    return np.stack(feats).astype(np.float32)


## 6. Extract Real Training Features

In [ ]:
real_paths = [os.path.join(TRAIN_IMG_DIR, f"{iid}.jpg") for iid in train_image_ids_all]
N_real = len(real_paths)

if cfg.use_flip4x:
    print(f"Extracting real features with 4-orientation augmentation ({N_real} × 4) ...")
    X_real         = extract_features(real_paths, augment=True,  log_prefix="real ")
    Y_real         = np.repeat(Y_all_real, 4, axis=0)
    real_image_ids = np.repeat(train_image_ids_all, 4)
    real_group_ids = np.repeat(np.arange(N_real), 4)
else:
    print(f"Extracting real features (no flip) for {N_real} images ...")
    X_real         = extract_features(real_paths, augment=False, log_prefix="real ")
    Y_real         = Y_all_real
    real_image_ids = train_image_ids_all
    real_group_ids = np.arange(N_real)

print(f"X_real: {X_real.shape}  Y_real: {Y_real.shape}  groups: {len(np.unique(real_group_ids))}")


## 7. Extract Test Features (once)

In [ ]:
df_test_raw = pd.read_csv(TEST_CSV)
df_test_raw["image_id"] = df_test_raw["sample_id"].str.split("__").str[0]
df_test_unique = df_test_raw.drop_duplicates("image_id").copy()

test_image_ids   = df_test_unique["image_id"].values
test_image_paths = [os.path.join(TEST_IMG_DIR, f"{iid}.jpg") for iid in test_image_ids]

print(f"Extracting features for {len(test_image_paths)} test images ...")
X_test = extract_features(test_image_paths, augment=False, log_prefix="test ")
print(f"X_test: {X_test.shape}")


## 8. DA-Fusion — Load Synthetic CSV & Extract Features (once)

Synthetic features are extracted **once**, then filtered per-fold against the
fold's training source-ids. The filter is the DA-Fusion leakage guard (only
keep synthetics whose `source_image` is in the current fold's train split).

If `USE_DA_FUSION=False`, this cell records empty synthetic arrays.

In [ ]:
if cfg.USE_DA_FUSION:
    def _source_id_from_path(p):
        return Path(str(p)).stem

    def _resolve_synth_path(image_path_field):
        candidates = [
            os.path.join(cfg.synthetic_dir, image_path_field),
            os.path.join(cfg.synthetic_dir, os.path.basename(image_path_field)),
            os.path.join(cfg.synthetic_dir, Path(image_path_field).name),
        ]
        for c in candidates:
            if os.path.exists(c):
                return c
        raise FileNotFoundError(
            f"Could not locate synthetic image: {image_path_field} under {cfg.synthetic_dir}"
        )

    df_synth_full = pd.read_csv(cfg.synthetic_csv)
    df_synth      = df_synth_full[df_synth_full["is_synthetic"].astype(bool)].copy()
    df_synth["source_id"] = df_synth["source_image"].apply(_source_id_from_path)
    print(f"Synthetic rows in CSV: {len(df_synth)}")

    synth_paths = [_resolve_synth_path(p) for p in df_synth["image_path"].values]
    print(f"Extracting DINOv2 features for {len(synth_paths)} synthetic images ...")
    X_synth_all = extract_features(synth_paths, augment=False, log_prefix="synth ")
    Y_synth_all = df_synth[TARGETS].values.astype(np.float32)
    synth_source_ids = df_synth["source_id"].values
    print(f"X_synth_all: {X_synth_all.shape}  Y_synth_all: {Y_synth_all.shape}")
else:
    X_synth_all      = np.zeros((0, cfg.input_dim), dtype=np.float32)
    Y_synth_all      = np.zeros((0, cfg.output_dim), dtype=np.float32)
    synth_source_ids = np.array([], dtype=object)
    print("USE_DA_FUSION=False — synthetic arrays empty.")


## 9. Model Architecture — BiomassModel (+ optional CHARMS i2t heads)

In [ ]:
class Encoder(nn.Module):
    """384 → 128 → 32 (GELU, Dropout). Matches baseline exactly."""
    def __init__(self, input_dim=384, latent_dim=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, latent_dim),
        )
    def forward(self, x):
        return self.net(x)


class Head(nn.Module):
    """32 → 32 → 5 (GELU, Dropout). Matches baseline exactly."""
    def __init__(self, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_dim),
        )
    def forward(self, z):
        return self.net(z)


class BiomassModel(nn.Module):
    """DINOv2 CLS → Encoder(384→128→32) → Head(32→32→5).

    When `use_charms_i2t=True`, also carries 32 linear i2t heads, each mapping
    a single latent scalar → 5-target prediction (partial CHARMS port).
    """
    def __init__(self, input_dim=384, latent_dim=32, output_dim=5, dropout=0.1,
                 use_charms_i2t=False):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim, dropout)
        self.head    = Head(latent_dim, output_dim, dropout)
        self.use_charms_i2t = use_charms_i2t
        self.latent_dim     = latent_dim
        if use_charms_i2t:
            # 32 linear heads, each Linear(1, 5) — one per latent dimension
            self.i2t_heads = nn.ModuleList(
                [nn.Linear(1, output_dim) for _ in range(latent_dim)]
            )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.head(self.encode(x))

    def forward_with_latent(self, x):
        z    = self.encode(x)
        pred = self.head(z)
        return pred, z

    def i2t_predict(self, z):
        """Return (B, latent_dim, output_dim) — per-latent i2t predictions."""
        assert self.use_charms_i2t, "i2t_heads not built"
        # z: (B, latent_dim).  Each head takes the scalar z[:, k:k+1].
        outs = [self.i2t_heads[k](z[:, k:k+1]) for k in range(self.latent_dim)]
        return torch.stack(outs, dim=1)   # (B, latent_dim, 5)

    def forward_cems(self, args, x, y_scaled, get_batch_cems_fn):
        """CEMS augmentation path. Encode, detach latent, CEMS-augment in 32-d.
        Returns (pred_aug, y_aug_scaled, latent_real). latent_real is returned
        (un-detached) so the caller can compute i2t loss on it.
        """
        latent_real = self.encode(x)
        latent_aug, y_aug_scaled = get_batch_cems_fn(
            args, latent_real.detach(), y_scaled, latent=True,
        )
        return self.head(latent_aug), y_aug_scaled, latent_real


_tmp = BiomassModel(
    cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout,
    use_charms_i2t=cfg.USE_CHARMS,
)
print(f"BiomassModel params: {sum(p.numel() for p in _tmp.parameters()):,} "
      f"(use_charms_i2t={cfg.USE_CHARMS})")
del _tmp


## 10. CEMS — Inlined Algorithm (port of `src/methods/cems/cems.py`)

Only used when `USE_CEMS=True`. Operates in the joint `[latent_32d, y_scaled]` space.

In [ ]:
def _twonn_batch(zi):
    with torch.no_grad():
        dists = torch.cdist(zi.float(), zi.float())
        dists.fill_diagonal_(float('inf'))
        top2 = dists.topk(2, dim=1, largest=False).values
        r1, r2 = top2[:, 0], top2[:, 1]
        mask = r1 > 0
        if mask.sum() < 3:
            return 2
        mu = (r2 / r1)[mask]
        mu_sorted, _ = mu.sort()
        n       = len(mu_sorted)
        ecdf    = torch.arange(1, n + 1, dtype=torch.float32, device=zi.device) / n
        log_mu  = torch.log(mu_sorted)
        log_surv = -torch.log1p(-ecdf + 1e-10)
        denom   = (log_mu * log_mu).sum()
        if denom.abs() < 1e-12:
            return 2
        d = float((log_mu * log_surv).sum() / denom)
    if not math.isfinite(d) or d < 1:
        return 2
    return max(1, int(round(d)))


def _adjust_dims(x, y, xk=None, yk=None):
    if x.ndim > 2: x = x.reshape(x.shape[0], -1)
    if y.ndim == 1: y = y.reshape(y.shape[0], 1)
    m  = x.shape[-1]
    zi = torch.cat((x, y), dim=-1)
    zk = None
    if xk is not None and yk is not None:
        if xk.ndim > 3:
            xk = xk.reshape(xk.shape[0], xk.shape[1], -1)
        zk = torch.cat((xk, yk), dim=-1)
    return x, zi, zk, m


def _solve_ridge(a, b, lam=1.0):
    n   = a.shape[-1]
    eye = torch.eye(n, device=a.device, dtype=a.dtype)
    a_t = a.transpose(-2, -1)
    return torch.linalg.inv(a_t @ a + lam * eye) @ a_t @ b


def _get_projection(args, x, xk):
    x_c = x.transpose(-2, -1)
    x_c_mean = None
    if args.cems_method == 1:
        x_c_mean = torch.mean(x_c, -1)
        x_c = x_c - x_c_mean.unsqueeze(-1)
    else:
        assert xk is not None
        xk_t = xk.transpose(-1, -2)
        x    = x.unsqueeze(-1)
        x_c  = xk_t - x

    svd_input  = (x_c - x_c_mean.unsqueeze(-1) if x_c_mean is not None else x_c)
    svd_kwargs = {"driver": "gesvd"} if svd_input.is_cuda else {}
    basis, _, _ = torch.linalg.svd(svd_input, full_matrices=False, **svd_kwargs)

    u      = basis.transpose(-2, -1) @ x_c
    u_prev = u.transpose(-2, -1)

    if args.cems_method == 1:
        u_t  = u.transpose(-1, -2)
        u    = (u_t.unsqueeze(1) - u_t).transpose(-1, -2)
        n    = x.shape[0]
        mask = ~torch.eye(n, dtype=torch.bool, device=x.device)
        u    = -u.transpose(-1, -2)[mask].reshape(
            (u.shape[0], u.shape[2] - 1, u.shape[1])
        ).transpose(-1, -2)
    elif args.cems_method == 2:
        u = u.unsqueeze(0)
    return basis, u, u_prev, x_c_mean


def _estimate_grad_hessian(args, x, xk, d):
    tidx      = torch.triu_indices(d, d, device=x.device)
    ones_mult = torch.ones((d, d), device=x.device)
    ones_mult.fill_diagonal_(0.5)
    basis, u, u_prev, x_mean = _get_projection(args, x, xk)
    u_d = u[:, :d]
    f   = u[:, d:].transpose(-2, -1)
    if args.use_hessian:
        uu = torch.einsum('bki,bkj->bkij',
                           u_d.transpose(-2, -1),
                           u_d.transpose(-2, -1))
        uu   = uu * ones_mult
        uu   = uu[:, :, tidx[0], tidx[1]].transpose(-2, -1)
        psi  = torch.cat((u_d, uu), dim=1).transpose(-2, -1)
    else:
        psi = u_d.transpose(-2, -1)
    lam    = torch.linalg.norm(psi, dim=(-1, -2)).mean()
    b_coef = _solve_ridge(psi, f, lam=lam).transpose(-2, -1)
    gradient = b_coef[..., :d]
    hessian  = torch.zeros((u.shape[0], b_coef.shape[1], d, d),
                            dtype=b_coef.dtype, device=b_coef.device)
    if args.use_hessian:
        hessian[..., tidx[0], tidx[1]] = b_coef[..., d:]
        hessian[..., tidx[1], tidx[0]] = b_coef[..., d:]
    return basis, gradient, hessian, u_d, u_prev, x_mean


def _sample_tangent(args, x, u_k_d, u_prev, x_mean, basis, grad, hess):
    d  = grad.shape[-1]
    nu = torch.distributions.Normal(0, args.sigma).sample(
        (x.shape[0], d, 1)
    ).to(x.device)
    f_nu = (grad @ nu).squeeze(-1)
    if args.use_hessian:
        nu_ex = nu.unsqueeze(1)
        f_nu  = f_nu + 0.5 * (nu_ex.transpose(-1, -2) @ hess @ nu_ex).squeeze((-1, -2))
    x_new_local = torch.cat((nu.squeeze(-1), f_nu), dim=-1)
    if args.cems_method == 1:
        x_new_local = x_new_local + u_prev
    x_cems = (basis @ x_new_local.unsqueeze(-1)).squeeze(-1)
    x_cems = x_cems + (x_mean if args.cems_method == 1 else x)
    return x_cems


def get_batch_cems(args, x, y, xk=None, yk=None, *, latent=True):
    x_shape, y_shape = x.shape, y.shape
    x_flat, zi, zk, m = _adjust_dims(x, y, xk, yk)
    d = args.id
    if latent:
        with torch.no_grad():
            d = _twonn_batch(zi)
    d = min(d, zi.shape[-1] - 1, zi.shape[-2] - 1)
    d = max(d, 1)
    basis, grad, hess, u_k_d, u_prev, x_mean = _estimate_grad_hessian(args, zi, zk, d)
    z_sampled = _sample_tangent(args, zi, u_k_d, u_prev, x_mean, basis, grad, hess)
    x_new = z_sampled[..., :m].reshape(x_shape)
    y_new = z_sampled[..., m:].reshape(y_shape)
    return x_new, y_new


def _next_power_of_2(n):
    if n < 1: return 1
    return 1 << (n - 1).bit_length()

def compute_neigh_size(d):
    base = d + d * (d + 1) // 2
    return _next_power_of_2(base + 1)

def precompute_knn(X, Y, neigh_size, compute_device="cpu"):
    XY = np.concatenate([X, Y], axis=1).astype(np.float32)
    t  = torch.tensor(XY, dtype=torch.float32)
    if compute_device in ("cuda", "mps"):
        try:
            t = t.to(compute_device)
        except Exception:
            pass
    dists = torch.cdist(t, t, p=2, compute_mode="donot_use_mm_for_euclid_dist")
    dists.fill_diagonal_(-float('inf'))
    sorted_idx = torch.argsort(dists, dim=1).cpu().numpy()
    return sorted_idx[:, :neigh_size].astype(np.int64)

print("CEMS functions defined.")


## 11. Loss, Metrics, Dataset, Eval

In [ ]:
WEIGHT_VECTOR = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float64)
_LOSS_WEIGHTS = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)

def _weighted_smooth_l1(pred, target, weights, beta=1.0):
    loss_per = nn.functional.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss_per * weights.to(pred.device)).mean()

def weighted_global_r2(y_true, y_pred):
    w  = WEIGHT_VECTOR
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)

def per_target_weighted_r2(y_true, y_pred):
    """Return {target: weighted_R²} and aggregate."""
    out = {}
    for i, t in enumerate(TARGETS):
        yt, yp = y_true[:, i], y_pred[:, i]
        ybar   = yt.mean()
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - ybar) ** 2) + 1e-12
        out[t] = float(1.0 - ss_res / ss_tot)
    return out

def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


class BiomassDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]


@torch.no_grad()
def eval_epoch(model, loader, eval_device):
    model.eval()
    total_loss, all_pred, all_true = 0.0, [], []
    for X, y in loader:
        X, y = X.to(eval_device), y.to(eval_device)
        pred = model(X)
        total_loss += _weighted_smooth_l1(pred, y, _LOSS_WEIGHTS).item() * len(X)
        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    return (
        total_loss / len(loader.dataset),
        weighted_global_r2(y_true, y_pred),
        rmse_per_target(y_true, y_pred),
        y_pred, y_true,
    )

print("Loss, metrics, dataset, eval_epoch defined.")


## 12. Per-Fold Training Function

Handles every toggle combination:
- `USE_DA_FUSION` → synthetic features appended to training set (per-fold leakage-filtered)
- `USE_CEMS` → anchor+kNN loop instead of mini-batch; real loss + augmented loss
- `USE_CHARMS` → i2t auxiliary loss added on the real-path latent (weight `cfg.charms_i2t_weight`)

In [ ]:
def _compute_charms_i2t_loss(model, latent, y_raw):
    """L_i2t = mean over 32 latent dims of weighted SmoothL1(i2t_head_k(z_k), y)."""
    preds = model.i2t_predict(latent)                # (B, latent_dim, 5)
    B, K, C = preds.shape
    y_exp   = y_raw.unsqueeze(1).expand(B, K, C)     # (B, K, 5)
    loss_el = nn.functional.smooth_l1_loss(preds, y_exp, reduction="none")  # (B, K, 5)
    w       = _LOSS_WEIGHTS.to(preds.device).view(1, 1, C)
    return (loss_el * w).mean()


def train_one_fold(
    X_train_np, Y_train_np, X_val_np, Y_val_np, cfg, fold_idx, verbose=True,
):
    """Train a BiomassModel on one fold. Returns (model, val_preds, val_true, history, wall_times)."""
    torch.manual_seed(cfg.seed + fold_idx)
    np.random.seed(cfg.seed + fold_idx)

    model = BiomassModel(
        cfg.input_dim, cfg.latent_dim, cfg.output_dim, cfg.dropout,
        use_charms_i2t=cfg.USE_CHARMS,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(cfg.NUM_EPOCHS, 1), eta_min=cfg.lr / 100
    )

    val_ds     = BiomassDataset(X_val_np, Y_val_np)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

    # Y scaler (for CEMS only) fit on this fold's training targets
    y_scaler = MinMaxScaler().fit(Y_train_np)
    Y_scaled = y_scaler.transform(Y_train_np).astype(np.float32)

    # kNN precomputation (CEMS only)
    neigh_size  = compute_neigh_size(cfg.initial_d) if cfg.USE_CEMS else 0
    knn_indices = (precompute_knn(X_train_np, Y_scaled, neigh_size,
                                   compute_device=device)
                   if cfg.USE_CEMS else None)

    # CEMS args
    cems_args = SimpleNamespace(
        sigma=cfg.sigma, cems_method=cfg.cems_method,
        id=cfg.initial_d, use_hessian=cfg.use_hessian,
    )

    X_t_full = torch.tensor(X_train_np, dtype=torch.float32, device=device)
    Y_t_full = torch.tensor(Y_train_np, dtype=torch.float32, device=device)
    Ys_t_full = torch.tensor(Y_scaled,   dtype=torch.float32, device=device)

    history = {"train_loss": [], "val_loss": [], "val_r2": []}
    wall_times = []
    best_val_r2 = -float("inf")
    best_state  = None

    mode = (
        f"CEMS={'L' if not cfg.use_hessian else 'full'}" if cfg.USE_CEMS
        else "ERM"
    )
    mode += f" + DA-Fusion" if cfg.USE_DA_FUSION else ""
    mode += f" + CHARMS-i2t(w={cfg.charms_i2t_weight})" if cfg.USE_CHARMS else ""
    if verbose:
        print(f"    mode: {mode}")
        print(f"    N_train={len(X_train_np)}  N_val={len(X_val_np)}  epochs={cfg.NUM_EPOCHS}")

    for epoch in range(1, cfg.NUM_EPOCHS + 1):
        t_start = time.time()
        model.train()
        ep_loss, n_steps = 0.0, 0

        if cfg.USE_CEMS:
            # ---- Anchor+kNN loop ----
            N         = len(X_train_np)
            samples   = np.random.permutation(N)
            if cfg.MAX_ANCHORS_PER_EPOCH is not None:
                samples = samples[:cfg.MAX_ANCHORS_PER_EPOCH]
            used = set()
            for anchor in samples:
                anchor = int(anchor)
                if anchor in used:
                    continue
                neigh_row  = knn_indices[anchor]
                candidates = neigh_row[1:]
                available  = [c for c in candidates if c not in used]
                n_neigh    = min(neigh_size - 1, len(available))
                if n_neigh == 0:
                    available = candidates[:neigh_size - 1].tolist()
                    n_neigh   = len(available)
                idx_all = np.concatenate([[anchor], np.array(available[:n_neigh], dtype=np.int64)])
                X_b   = X_t_full[idx_all]
                Y_b   = Y_t_full[idx_all]
                Ys_b  = Ys_t_full[idx_all]

                optimizer.zero_grad()

                # Real path + latent (for CHARMS i2t)
                pred_real, latent_real = model.forward_with_latent(X_b)
                loss_real = _weighted_smooth_l1(pred_real, Y_b, _LOSS_WEIGHTS)

                # CEMS augmented path
                pred_aug, ys_aug, _ = model.forward_cems(
                    cems_args, X_b, Ys_b, get_batch_cems,
                )
                y_aug_np  = y_scaler.inverse_transform(
                    ys_aug.detach().cpu().numpy()
                ).astype(np.float32)
                y_aug_raw = torch.tensor(y_aug_np, dtype=torch.float32, device=device)
                loss_aug  = _weighted_smooth_l1(pred_aug, y_aug_raw, _LOSS_WEIGHTS)

                loss = loss_real + loss_aug

                if cfg.USE_CHARMS:
                    loss = loss + cfg.charms_i2t_weight * _compute_charms_i2t_loss(
                        model, latent_real, Y_b,
                    )

                loss.backward()
                optimizer.step()
                ep_loss += loss.item() * len(idx_all)
                n_steps += len(idx_all)
                used.update(idx_all.tolist())
        else:
            # ---- Standard mini-batch loop ----
            train_ds     = BiomassDataset(X_train_np, Y_train_np)
            train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                                       shuffle=True, num_workers=0, drop_last=False)
            for X_b, Y_b in train_loader:
                X_b, Y_b = X_b.to(device), Y_b.to(device)
                optimizer.zero_grad()
                pred, latent = model.forward_with_latent(X_b)
                loss = _weighted_smooth_l1(pred, Y_b, _LOSS_WEIGHTS)
                if cfg.USE_CHARMS:
                    loss = loss + cfg.charms_i2t_weight * _compute_charms_i2t_loss(
                        model, latent, Y_b,
                    )
                loss.backward()
                optimizer.step()
                ep_loss += loss.item() * len(X_b)
                n_steps += len(X_b)

        scheduler.step()

        tr           = ep_loss / max(n_steps, 1)
        val_loss, val_r2, val_rmse, _, _ = eval_epoch(model, val_loader, device)
        wall_times.append(time.time() - t_start)

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history["train_loss"].append(tr)
        history["val_loss"].append(val_loss)
        history["val_r2"].append(val_r2)

        if verbose and (epoch == 1 or epoch % 10 == 0 or epoch == cfg.NUM_EPOCHS):
            print(f"      ep {epoch:3d}  tr={tr:.4f}  val_loss={val_loss:.4f}  "
                  f"val_R²={val_r2:.4f}  wall={wall_times[-1]:.1f}s")

    # Restore best
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    _, val_r2_final, val_rmse_final, val_preds, val_true = eval_epoch(model, val_loader, device)
    return model, val_preds, val_true, history, wall_times, val_r2_final, val_rmse_final

print("train_one_fold defined.")


## 13. GroupKFold CV Loop — Run All Folds

Splits the real features on `real_group_ids` (baseline-compatible), filters
synthetics per fold by `source_id ∈ train_source_ids`, trains the model, and
records OOF predictions + test predictions.

Per-fold output reports weighted R² per target + aggregate, plus fold-0 epoch
wall-clock time for the smoke test.

In [ ]:
os.makedirs(cfg.output_dir, exist_ok=True)

gkf = GroupKFold(n_splits=5)   # fixed at 5 to preserve the baseline split
all_splits = list(gkf.split(X_real, groups=real_group_ids))
active_splits = all_splits[:cfg.NUM_FOLDS]

# Map source_id → group id (for synthetic filtering)
source_to_group = {iid: int(grp)
                   for iid, grp in zip(real_image_ids, real_group_ids)}

# OOF predictions over real val rows
oof_preds    = np.zeros_like(Y_real, dtype=np.float32)
oof_mask     = np.zeros(len(Y_real), dtype=bool)
fold_test_preds = []
fold_summaries  = []

SMOKE_LOG_TAG = "[SMOKE]" if cfg.SMOKE_TEST else ""
print(f"{SMOKE_LOG_TAG} Running {cfg.NUM_FOLDS} fold(s) with epochs={cfg.NUM_EPOCHS} "
      f"USE_DA_FUSION={cfg.USE_DA_FUSION} USE_CEMS={cfg.USE_CEMS} USE_CHARMS={cfg.USE_CHARMS}")

for fold_idx, (train_idx, val_idx) in enumerate(active_splits):
    print(f"\n===== Fold {fold_idx + 1}/{cfg.NUM_FOLDS} =====")

    # ---- Leakage-safe split ----
    train_groups = set(real_group_ids[train_idx].tolist())
    val_groups   = set(real_group_ids[val_idx].tolist())
    assert train_groups.isdisjoint(val_groups), "Group leakage in real split!"
    train_source_ids = set(real_image_ids[train_idx].tolist())

    X_tr_real = X_real[train_idx]; Y_tr_real = Y_real[train_idx]
    X_vl      = X_real[val_idx];   Y_vl      = Y_real[val_idx]

    # ---- DA-Fusion: per-fold leakage filter ----
    if cfg.USE_DA_FUSION and len(X_synth_all) > 0:
        synth_mask = np.array([sid in train_source_ids for sid in synth_source_ids])
        X_tr_synth = X_synth_all[synth_mask]
        Y_tr_synth = Y_synth_all[synth_mask]
        X_tr = np.vstack([X_tr_real, X_tr_synth]).astype(np.float32)
        Y_tr = np.vstack([Y_tr_real, Y_tr_synth]).astype(np.float32)
        # Sanity: no synthetic sources leak into val
        kept_sources = set(synth_source_ids[synth_mask].tolist())
        val_source_ids = set(real_image_ids[val_idx].tolist())
        assert kept_sources.isdisjoint(val_source_ids), \
            "DA-Fusion leakage: synthetic source is in val fold!"
        print(f"  real_train={len(X_tr_real)}  synth_kept={len(X_tr_synth)} "
              f"(of {len(X_synth_all)} total)  total_train={len(X_tr)}")
    else:
        X_tr, Y_tr = X_tr_real.astype(np.float32), Y_tr_real.astype(np.float32)
        print(f"  real_train={len(X_tr_real)}  (no synthetics)")

    # ---- Train ----
    model, val_preds, val_true, history, wall_times, val_r2_final, val_rmse = train_one_fold(
        X_tr, Y_tr, X_vl, Y_vl, cfg, fold_idx, verbose=True,
    )

    # ---- Fold-0 wall-clock warning (smoke-test requirement) ----
    if fold_idx == 0 and len(wall_times) > 0:
        avg_wall = np.mean(wall_times)
        max_wall = np.max(wall_times)
        print(f"  [fold 0] epoch wall-clock — mean={avg_wall:.1f}s  max={max_wall:.1f}s")
        if max_wall > 600.0:
            print("  *** WARNING: fold-0 epoch exceeded 10 minutes. "
                  "Consider cfg.MAX_ANCHORS_PER_EPOCH or reducing synthetic ratio.")

    # ---- OOF + test predictions ----
    oof_preds[val_idx] = val_preds
    oof_mask[val_idx]  = True

    model.eval()
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
    with torch.no_grad():
        tp = model(X_test_t).cpu().numpy()
    tp = np.clip(tp, 0.0, None)
    fold_test_preds.append(tp)

    # ---- Per-fold reporting ----
    per_tgt_r2 = per_target_weighted_r2(val_true, val_preds)
    print(f"  >>> Fold {fold_idx} weighted R² (aggregate): {val_r2_final:.4f}")
    print(f"      Per-target R²:")
    for t in TARGETS:
        print(f"        {t:<16}: R²={per_tgt_r2[t]:+.4f}  RMSE={val_rmse[t]:.3f}")

    fold_summaries.append({
        "fold":         fold_idx,
        "val_r2":       float(val_r2_final),
        "per_target_r2": per_tgt_r2,
        "per_target_rmse": {k: float(v) for k, v in val_rmse.items()},
        "epoch_wall_times": [float(x) for x in wall_times],
    })

print("\n===== CV complete =====")


## 14. Aggregate OOF Metrics & Persist to Disk

In [ ]:
if oof_mask.sum() > 0:
    oof_r2    = weighted_global_r2(Y_real[oof_mask], oof_preds[oof_mask])
    oof_per_t = per_target_weighted_r2(Y_real[oof_mask], oof_preds[oof_mask])
    oof_rmse  = rmse_per_target(Y_real[oof_mask], oof_preds[oof_mask])
    print(f"OOF weighted R² (aggregate, {oof_mask.sum()} rows): {oof_r2:.4f}")
    for t in TARGETS:
        print(f"  {t:<16}: R²={oof_per_t[t]:+.4f}  RMSE={oof_rmse[t]:.3f}")
else:
    oof_r2 = float('nan')
    print("No OOF coverage (NUM_FOLDS=1 with full CV would be needed).")

# Save OOF predictions (for ensemble_charms.py)
oof_out = os.path.join(cfg.output_dir, "oof_combined.npz")
np.savez(
    oof_out,
    oof_preds     = oof_preds,
    oof_mask      = oof_mask,
    y_true        = Y_real,
    real_image_ids = real_image_ids.astype(str),
    real_group_ids = real_group_ids,
    config        = np.array(json.dumps({
        "USE_DA_FUSION": cfg.USE_DA_FUSION,
        "USE_CEMS":      cfg.USE_CEMS,
        "USE_CHARMS":    cfg.USE_CHARMS,
        "NUM_EPOCHS":    cfg.NUM_EPOCHS,
        "NUM_FOLDS":     cfg.NUM_FOLDS,
        "SMOKE_TEST":    cfg.SMOKE_TEST,
        "oof_r2":        oof_r2,
    })),
)
print(f"OOF saved to: {oof_out}")

# Save fold summaries as JSON
fold_json = os.path.join(cfg.output_dir, "fold_summaries.json")
with open(fold_json, "w") as f:
    json.dump({"oof_r2": oof_r2, "folds": fold_summaries}, f, indent=2)
print(f"Fold summaries saved to: {fold_json}")


## 15. Test Inference — Average Across Folds

Test predictions are averaged across the `NUM_FOLDS` fold models (simple mean).

In [ ]:
test_preds_mean = np.mean(np.stack(fold_test_preds, axis=0), axis=0)
test_preds_mean = np.clip(test_preds_mean, 0.0, None)
print(f"test_preds_mean: {test_preds_mean.shape}  range=[{test_preds_mean.min():.2f}, {test_preds_mean.max():.2f}]")
for i in range(min(3, len(test_image_ids))):
    print(f"  {test_image_ids[i]}: {dict(zip(TARGETS, test_preds_mean[i].round(2)))}")


## 16. Write Submission CSV — `submission_combined.csv`

Same long-format layout as baseline (columns: `sample_id`, `target`).

In [ ]:
def prepare_submission(test_csv_path, predictions, image_ids):
    df_t = pd.read_csv(test_csv_path)
    pred_dict = {
        img_id: {col: float(val) for col, val in zip(TARGETS, pred_row)}
        for img_id, pred_row in zip(image_ids, predictions)
    }
    def _get_pred(row):
        img_id      = row["sample_id"].split("__")[0]
        target_name = row["target_name"]
        return max(0.0, pred_dict.get(img_id, {}).get(target_name, 0.0))
    df_t["target"] = df_t.apply(_get_pred, axis=1)
    return df_t[["sample_id", "target"]]

submission = prepare_submission(TEST_CSV, test_preds_mean, test_image_ids)
out_path = os.path.join(cfg.output_dir, "submission_combined.csv")
submission.to_csv(out_path, index=False)

print(f"Submission saved to: {out_path}")
print(f"Rows: {len(submission)}")
print(submission.head(10))
print("\nTarget value stats:")
print(submission["target"].describe().round(3))
print("\nWorking dir:", os.listdir(cfg.output_dir))
